In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [4]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    theta_adv = deepcopy(theta_0)
    for i in range(X_0.shape[1]):
        theta_r_min = deepcopy(theta_adv)
        theta_r_max = deepcopy(theta_adv)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha
    
    return theta_adv

In [5]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    i_max = 0
    alpha_max = 0
    val_max = -np.inf

    for i in range(X_0.shape[1]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        if np.mean(J_min) > np.mean(J_max):
            if np.mean(J_min) > val_max:
                i_max = i
                alpha_max = -alpha
                val_max = np.mean(J_min).item()
        else:
            if np.mean(J_max) > val_max:
                i_max = i
                alpha_max = alpha
                val_max = np.mean(J_max).item()
    
    theta_adv = deepcopy(theta_0)
    theta_adv[i_max] = alpha_max
    return theta_adv

In [73]:
def evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    
    weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [74]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [75]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    weights_0, bias_0 = theta_0[:-1], theta_0[[-1]]
    # Don't need this block of code below
    if theta_adv_method=='L-1':
        theta_adv = get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb)
    else:
        theta_adv = get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb)
    # weights_adv, bias_adv = theta_adv[:-1], theta_adv[[-1]]
        
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf.model.coef_ = weights_0.reshape(1,-1)
    clf.model.intercept_ = bias_0
    
    clf_adv = deepcopy(clf)
    # clf_adv.model.coef_ = weights_adv.reshape(1,-1)
    # clf_adv.model.intercept_ = bias_adv

    for i in tqdm.trange(n, desc=f'[{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]

        weights_adv, bias_adv = calTheta(x_r, weights_0, bias_0, alpha, theta_adv_method)
        clf_adv.model.coef_ = weights_adv.reshape(1,-1)
        clf_adv.model.intercept_ = bias_adv

        J = RecourseCost(x_0, lamb)
        
        bce_loss, cost, price = J.eval(x_r, weights_adv, bias_adv, True)
        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, theta_0, theta_adv)

In [89]:
alphas = np.linspace(0,0.5,5)
lambdas = [0.1,0.3]


params = {}
# 'synthetic', 'german', 'sba'
params['data'] = 'sba'
# 'lr', 'nn'
params['base_model'] = 'lr'
params['seeds'] = range(5)
# TODO: add your method here, the method name should match the name in filepath
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# 'one', 'many'
params['adv_method'] = 'many'

results = {
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
}

for algorithm in params['algorithms']:
    for seed in params['seeds']:
        for v_alpha in alphas:
            for v_lamb in lambdas:
                data = pd.read_pickle(f"../results/recourse/{params['base_model']}_{params['data']}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                alpha = data["alpha"].unique().item()
                lamb = data["lambda"].unique().item()
                theta_0 = data["theta_0"].iloc[0]
                weights_0, bias_0, = theta_0[:-1], theta_0[[-1]]
                X_0 = np.stack(data["x_0"])
                X_r = np.stack(data["x_r"])
                if params['adv_method'] == 'one':
                    res = evaluate_performance(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                else:
                    res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                results['algorithm'].append(algorithm)
                results['seed'].append(seed)
                results['Cost'].append(res['cost'])
                results['Current Validity'].append(res['m1_probability'])
                results['Worst Case Validity'].append(res['wc_probability'])
                results['alpha'].append(alpha)
                results['lambda'].append(lamb)

df_results = pd.DataFrame(results)

[Alg1] [ seed=0 ] [ α=0.0 ] [ λ=0.1 ]: 100%|██████████| 39/39 [00:00<00:00, 2104.87it/s]


[Roarl1] [ seed=4 ] [ α=0.5 ] [ λ=0.3 ]: 100%|██████████| 38/38 [00:00<00:00, 2303.36it/s]


In [90]:
df_graph = df_results.groupby(["alpha", "lambda", "algorithm"]).mean().reset_index()
mask = df_graph['algorithm'] == 'Alg1'
df_graph.loc[mask,'algorithm'] = 'LInf'
df_graph['algorithm_lamb'] = df_graph[['algorithm', 'lambda']].apply(lambda row: row['algorithm'] + '(Lamb = ' + str(row['lambda']) + ')' , axis=1)


df_graph

,alpha,lambda,algorithm,seed,Cost,Current Validity,Worst Case Validity,algorithm_lamb
0,0.000,0.1,LInf,2.0,2.384078,0.969722,0.969722,LInf(Lamb = 0.1)
1,0.000,0.1,L1PSD,2.0,2.585275,0.969727,0.969727,L1PSD(Lamb = 0.1)
2,0.000,0.1,ROARL1,2.0,2.927102,0.869950,0.869950,ROARL1(Lamb = 0.1)
3,0.000,0.1,ROARLInf,2.0,2.913447,0.870113,0.870113,ROARLInf(Lamb = 0.1)
4,0.000,0.3,LInf,2.0,2.031897,0.909165,0.909165,LInf(Lamb = 0.3)
5,0.000,0.3,L1PSD,2.0,2.363156,0.909259,0.909259,L1PSD(Lamb = 0.3)
6,0.000,0.3,ROARL1,2.0,1.949708,0.559417,0.559417,ROARL1(Lamb = 0.3)
7,0.000,0.3,ROARLInf,2.0,1.949708,0.559417,0.559417,ROARLInf(Lamb = 0.3)
8,0.125,0.1,LInf,2.0,2.961065,0.995045,0.968562,LInf(Lamb = 0.1)
9,0.125,0.1,L1PSD,2.0,2.664194,0.975558,0.969697,L1PSD(Lamb = 0.1)


In [91]:
# custom_colors = {
#     "Alg1(0.1)": "#009999",
#     "L1PSD(0.1)": "#990000",
#     "ROARL1(0.1)": "#009900",
#     "ROARLInf(0.1)": "#990099",
#     "Alg1(0.3)": "#33FFFF",
#     "L1PSD(0.3)": "#FF3333",
#     "ROARL1(0.3)": "#33FF33",
#     "ROARLInf(0.3)": "#FF33FF"
# }

custom_colors = {
    "LInf(Lamb = 0.1)": "#33FFFF",
    "L1PSD(Lamb = 0.1)": "#FF3333",
    "ROARL1(Lamb = 0.1)": "#33FF33",
    "ROARLInf(Lamb = 0.1)": "#FF33FF",
    "LInf(Lamb = 0.3)": "#33FFFF",
    "L1PSD(Lamb = 0.3)": "#FF3333",
    "ROARL1(Lamb = 0.3)": "#33FF33",
    "ROARLInf(Lamb = 0.3)": "#FF33FF"
}
fig = px.line(df_graph, 
           x="Cost", y="Worst Case Validity", 
           color="algorithm_lamb",
           hover_data=["alpha", "lambda"],
           title=f"{params['data']}_data",
           markers=True,
           color_discrete_map=custom_colors,
           facet_col="lambda") 
fig

if params['adv_method'] == "one":
    fig.write_html(f"{params['data']}_costValidityTradeoff_oneAdv.html")
else:
    fig.write_html(f"{params['data']}_costValidityTradeoff_manyAdv.html")
fig
    